# 🏎️ Formula 1 Historical Analysis: The GOAT Debate & Era Comparisons

**Author:** F1 Data Analyst  
**Date:** 2025  
**Dataset:** F1 Results (1950 - Present)

This notebook provides a comprehensive analysis of Formula 1 history, including:
- 🏆 Greatest of All Time (GOAT) Analysis
- 📊 Era Comparisons (1950s to Present)
- 📈 Championship Statistics & Trends
- 🎯 Fan Recommendations

---

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✅ Libraries loaded successfully!")

✅ Libraries loaded successfully!


## 1. Data Loading & Exploration

In [2]:
# Load the F1 results data
df = pd.read_csv('f1_results.csv')

# Display basic info
print("📊 Dataset Shape:", df.shape)
print("\n📅 Season Range:", df['season'].min(), "-", df['season'].max())
print("\n🏁 Total Races:", df[['season', 'round']].drop_duplicates().shape[0])
print("\n👨‍🏎️ Unique Drivers:", df['driver_name'].nunique())
print("\n🏭 Unique Constructors:", df['constructor'].nunique())

# Display first few rows
df.head(10)

📊 Dataset Shape: (25939, 15)

📅 Season Range: 1950 - 2026

🏁 Total Races: 1152

👨‍🏎️ Unique Drivers: 818

🏭 Unique Constructors: 205


,season,round,race_name,driver_id,driver_name,constructor_id,constructor,grid,position,points,laps,status,time,fastest_lap,fastest_lap_rank
0,1950,1,British Grand Prix,farina,Nino Farina,alfa,Alfa Romeo,1,1,9.0,70,Finished,2:13:23.600,NaN,NaN
1,1950,1,British Grand Prix,fagioli,Luigi Fagioli,alfa,Alfa Romeo,2,2,6.0,70,Finished,+2.600,NaN,NaN
2,1950,1,British Grand Prix,reg_parnell,Reg Parnell,alfa,Alfa Romeo,4,3,4.0,70,Finished,+52.000,NaN,NaN
3,1950,1,British Grand Prix,cabantous,Yves Cabantous,lago,Talbot-Lago,6,4,3.0,68,+2 Laps,NaN,NaN,NaN
4,1950,1,British Grand Prix,rosier,Louis Rosier,lago,Talbot-Lago,9,5,2.0,68,+2 Laps,NaN,NaN,NaN
5,1950,1,British Grand Prix,gerard,Bob Gerard,era,ERA,13,6,0.0,67,+3 Laps,NaN,NaN,NaN
6,1950,1,British Grand Prix,harrison,Cuth Harrison,era,ERA,15,7,0.0,67,+3 Laps,NaN,NaN,NaN
7,1950,1,British Grand Prix,etancelin,Philippe Étancelin,lago,Talbot-Lago,14,8,0.0,65,+5 Laps,NaN,NaN,NaN
8,1950,1,British Grand Prix,hampshire,David Hampshire,maserati,Maserati,16,9,0.0,64,+6 Laps,NaN,NaN,NaN
9,1950,1,British Grand Prix,fry,Joe Fry,maserati,Maserati,20,10,0.0,64,+6 Laps,NaN,NaN,NaN


In [3]:
# Data quality check
print("\n🔍 Data Quality Check:")
print(df.isnull().sum())

# Convert position to numeric (handling non-finishers)
df['position_num'] = pd.to_numeric(df['position'], errors='coerce')
df['is_winner'] = df['position_num'] == 1
df['is_podium'] = df['position_num'] <= 3

print("\n✅ Data preprocessing complete!")


🔍 Data Quality Check:
season                  0
round                   0
race_name               0
driver_id               0
driver_name             0
constructor_id          0
constructor             0
grid                    0
position                0
points                  0
laps                    0
status                  0
time                17585
fastest_lap         17154
fastest_lap_rank    17154
dtype: int64

✅ Data preprocessing complete!


## 2. 🏆 GOAT Analysis: Greatest Drivers of All Time

We'll analyze drivers across multiple dimensions to determine the GOAT.

In [4]:
# Calculate comprehensive driver statistics
driver_stats = df.groupby('driver_name').agg({
    'season': ['min', 'max', 'nunique'],
    'race_name': 'nunique',
    'is_winner': 'sum',
    'is_podium': 'sum',
    'points': 'sum',
    'grid': 'mean',
    'position_num': 'mean'
}).round(2)

# Flatten column names
driver_stats.columns = ['career_start', 'career_end', 'seasons_active', 
                        'races_entered', 'wins', 'podiums', 'total_points', 
                        'avg_grid', 'avg_finish']

# Calculate additional metrics
driver_stats['career_span'] = driver_stats['career_end'] - driver_stats['career_start'] + 1
driver_stats['win_rate'] = (driver_stats['wins'] / driver_stats['races_entered'] * 100).round(2)
driver_stats['podium_rate'] = (driver_stats['podiums'] / driver_stats['races_entered'] * 100).round(2)
driver_stats['points_per_race'] = (driver_stats['total_points'] / driver_stats['races_entered']).round(2)

# Filter drivers with at least 10 races for meaningful stats
qualified_drivers = driver_stats[driver_stats['races_entered'] >= 10].copy()

print(f"📊 Analyzed {len(qualified_drivers)} drivers with 10+ races")
qualified_drivers.head(10)

📊 Analyzed 303 drivers with 10+ races


,career_start,career_end,seasons_active,races_entered,wins,podiums,total_points,avg_grid,avg_finish,career_span,win_rate,podium_rate,points_per_race
driver_name,,,,,,,,,,,,,
Adrian Sutil,2007,2014,7,24,0,0,124.00,14.80,14.52,8,0.00,0.00,5.17
Adrián Campos,1987,1988,2,16,0,0,0.00,19.56,19.72,2,0.00,0.00,0.00
Aguri Suzuki,1988,1995,7,20,0,1,8.00,16.53,16.18,8,0.00,5.00,0.40
Alain Prost,1980,1993,13,26,51,106,798.50,4.14,7.50,14,196.15,407.69,30.71
Alan Jones,1975,1986,10,25,12,24,206.00,11.49,11.40,12,48.00,96.00,8.24
Alberto Ascari,1950,1955,6,11,13,17,140.14,4.58,8.44,6,118.18,154.55,12.74
Alessandro Nannini,1986,1990,5,18,1,9,65.00,12.52,14.81,5,5.56,50.00,3.61
Alessandro Zanardi,1991,1999,5,17,0,0,1.00,16.38,15.60,9,0.00,0.00,0.06
Alex Caffi,1986,1991,6,18,0,0,6.00,18.84,14.80,6,0.00,0.00,0.33


In [5]:
# Top 20 Drivers by Wins
top_winners = qualified_drivers.nlargest(20, 'wins')

fig = px.bar(
    top_winners.reset_index(),
    x='wins',
    y='driver_name',
    orientation='h',
    title='🏆 Top 20 F1 Drivers by Race Wins',
    color='wins',
    color_continuous_scale='RdYlGn',
    text='wins',
    hover_data=['win_rate', 'podiums', 'career_span']
)

fig.update_layout(
    yaxis=dict(autorange="reversed"),
    xaxis_title="Race Wins",
    yaxis_title="Driver",
    height=700,
    template='plotly_dark'
)
fig.show()

In [6]:
# Win Rate vs Total Wins Scatter Plot
# Filter for drivers with at least 50 races for meaningful win rate
experienced_drivers = qualified_drivers[qualified_drivers['races_entered'] >= 50].copy()

fig = px.scatter(
    experienced_drivers.reset_index(),
    x='races_entered',
    y='win_rate',
    size='wins',
    color='wins',
    hover_name='driver_name',
    title='📈 Win Rate vs Races Entered (Bubble size = Total Wins)',
    color_continuous_scale='Viridis',
    hover_data=['podium_rate', 'career_span', 'total_points']
)

# Add annotations for top drivers
top_5_drivers = experienced_drivers.nlargest(5, 'wins').index
for driver in top_5_drivers:
    row = experienced_drivers.loc[driver]
    fig.add_annotation(
        x=row['races_entered'],
        y=row['win_rate'],
        text=driver,
        showarrow=True,
        arrowhead=2,
        ax=40,
        ay=-40
    )

fig.update_layout(
    xaxis_title="Races Entered",
    yaxis_title="Win Rate (%)",
    height=600,
    template='plotly_dark'
)
fig.show()

In [7]:
# GOAT Score Calculation
# Normalize metrics (0-100 scale) and create composite score

def normalize_metric(series, higher_is_better=True):
    """Normalize series to 0-100 scale"""
    if higher_is_better:
        return ((series - series.min()) / (series.max() - series.min()) * 100).fillna(0)
    else:
        return ((series.max() - series) / (series.max() - series.min()) * 100).fillna(0)

# Calculate GOAT scores for drivers with 30+ races
goat_candidates = qualified_drivers[qualified_drivers['races_entered'] >= 30].copy()

# Normalize key metrics
goat_candidates['wins_score'] = normalize_metric(goat_candidates['wins'])
goat_candidates['win_rate_score'] = normalize_metric(goat_candidates['win_rate'])
goat_candidates['podiums_score'] = normalize_metric(goat_candidates['podiums'])
goat_candidates['podium_rate_score'] = normalize_metric(goat_candidates['podium_rate'])
goat_candidates['points_score'] = normalize_metric(goat_candidates['total_points'])
goat_candidates['longevity_score'] = normalize_metric(goat_candidates['career_span'])

# Calculate composite GOAT score (weighted average)
weights = {
    'wins': 0.25,
    'win_rate': 0.20,
    'podiums': 0.20,
    'podium_rate': 0.15,
    'points': 0.10,
    'longevity': 0.10
}

goat_candidates['goat_score'] = (
    goat_candidates['wins_score'] * weights['wins'] +
    goat_candidates['win_rate_score'] * weights['win_rate'] +
    goat_candidates['podiums_score'] * weights['podiums'] +
    goat_candidates['podium_rate_score'] * weights['podium_rate'] +
    goat_candidates['points_score'] * weights['points'] +
    goat_candidates['longevity_score'] * weights['longevity']
).round(2)

# Display top 15 GOAT candidates
top_goat = goat_candidates.nlargest(15, 'goat_score')[
    ['wins', 'win_rate', 'podiums', 'podium_rate', 'total_points', 'career_span', 'goat_score']
]

print("🏆 TOP 15 GOAT CANDIDATES:")
print("="*80)
top_goat

🏆 TOP 15 GOAT CANDIDATES:


,wins,win_rate,podiums,podium_rate,total_points,career_span,goat_score
driver_name,,,,,,,
Lewis Hamilton,105,269.23,203,520.51,4990.5,20,95.14
Michael Schumacher,91,303.33,155,516.67,1566.0,22,83.21
Max Verstappen,71,186.84,127,334.21,3313.5,12,61.91
Sebastian Vettel,53,135.90,122,312.82,3098.0,16,54.47
Fernando Alonso,32,86.49,106,286.49,2380.0,26,46.78
Kimi Räikkönen,21,55.26,103,271.05,1873.0,21,38.17
Valtteri Bottas,10,25.00,67,167.50,1788.0,14,23.81
Charles Leclerc,8,22.22,52,144.44,1630.0,9,18.52
Sergio Pérez,6,15.38,39,100.00,1585.0,16,17.98


In [8]:
# GOAT Score Visualization
fig = px.bar(
    top_goat.reset_index(),
    x='goat_score',
    y='driver_name',
    orientation='h',
    title='🏆 GOAT Score Rankings (Top 15)',
    color='goat_score',
    color_continuous_scale='RdBu',
    text='goat_score',
    hover_data=['wins', 'win_rate', 'podiums']
)

fig.update_layout(
    yaxis=dict(autorange="reversed"),
    xaxis_title="GOAT Score (0-100)",
    yaxis_title="Driver",
    height=600,
    template='plotly_dark'
)
fig.show()

## 3. 📊 Era Comparisons: Evolution of F1

Analyze how F1 has changed across different eras.

In [9]:
# Define F1 Eras
def get_era(season):
    if season < 1960:
        return "Pioneer Era (1950-1959)"
    elif season < 1970:
        return "Golden Era (1960-1969)"
    elif season < 1980:
        return "Turbo Era (1970-1979)"
    elif season < 1990:
        return "Turbo Era (1980-1989)"
    elif season < 2000:
        return "V10 Era (1990-1999)"
    elif season < 2010:
        return "V10/V8 Era (2000-2009)"
    elif season < 2014:
        return "V8 Era (2010-2013)"
    elif season < 2022:
        return "Hybrid Era (2014-2021)"
    else:
        return "Ground Effect Era (2022+)"

df['era'] = df['season'].apply(get_era)

# Era statistics
era_stats = df.groupby('era').agg({
    'race_name': 'nunique',
    'driver_name': 'nunique',
    'constructor': 'nunique',
    'is_winner': 'sum',
    'points': 'mean'
}).round(2)

era_stats.columns = ['total_races', 'unique_drivers', 'unique_constructors', 'total_wins', 'avg_points']
era_stats['races_per_season'] = (era_stats['total_races'] / 
    df.groupby('era')['season'].nunique()).round(1)

print("📊 ERA STATISTICS:")
print("="*80)
era_stats

📊 ERA STATISTICS:


,total_races,unique_drivers,unique_constructors,total_wins,avg_points,races_per_season
era,,,,,,
Golden Era (1960-1969),16,214,69,100,1.21,1.6
Ground Effect Era (2022+),25,32,14,95,5.07,5.0
Hybrid Era (2014-2021),36,51,19,160,4.98,4.5
Pioneer Era (1950-1959),15,313,67,87,1.03,1.5
Turbo Era (1970-1979),18,156,52,144,1.03,1.8
Turbo Era (1980-1989),26,104,31,156,0.98,2.6
V10 Era (1990-1999),23,97,31,162,1.07,2.3
V10/V8 Era (2000-2009),22,71,23,174,1.68,2.2
V8 Era (2010-2013),21,42,15,77,4.31,5.2


In [10]:
# Races per Season by Era
races_per_season = df.groupby(['era', 'season'])['race_name'].nunique().reset_index()
races_per_season.columns = ['era', 'season', 'race_count']

fig = px.box(
    races_per_season,
    x='era',
    y='race_count',
    title='📦 Races per Season Distribution by Era',
    color='era',
    points='all',
    hover_data=['season']
)

fig.update_layout(
    xaxis_title="Era",
    yaxis_title="Number of Races",
    height=500,
    template='plotly_dark',
    showlegend=False
)
fig.update_xaxes(tickangle=45)
fig.show()

In [11]:
# Evolution of Race Wins Over Time
wins_by_year = df[df['is_winner']].groupby('season').size().reset_index(name='wins')
wins_by_year['era'] = wins_by_year['season'].apply(get_era)

fig = px.scatter(
    wins_by_year,
    x='season',
    y='wins',
    color='era',
    title='🏁 Race Wins Distribution Across F1 History',
    size='wins',
    hover_data=['wins']
)

# Add trend line
fig.add_trace(go.Scatter(
    x=wins_by_year['season'],
    y=wins_by_year['wins'].rolling(window=5, center=True).mean(),
    mode='lines',
    name='5-Year Trend',
    line=dict(color='white', width=2, dash='dash')
))

fig.update_layout(
    xaxis_title="Season",
    yaxis_title="Number of Races",
    height=500,
    template='plotly_dark'
)
fig.show()

In [12]:
# Constructor Dominance by Era
constructor_era = df[df['is_winner']].groupby(['era', 'constructor']).size().reset_index(name='wins')
constructor_era = constructor_era.sort_values(['era', 'wins'], ascending=[True, False])

# Get top constructor per era
top_constructor_per_era = constructor_era.groupby('era').first().reset_index()

fig = px.bar(
    top_constructor_per_era,
    x='era',
    y='wins',
    color='constructor',
    title='🏭 Most Successful Constructor by Era',
    text='constructor',
    hover_data=['wins']
)

fig.update_layout(
    xaxis_title="Era",
    yaxis_title="Wins",
    height=500,
    template='plotly_dark',
    xaxis_tickangle=45
)
fig.show()

## 4. 🏁 Championship Statistics

In [13]:
# Calculate season champions (most wins per season)
season_winners = df[df['is_winner']].groupby(['season', 'driver_name']).size().reset_index(name='wins')
champions = season_winners.loc[season_winners.groupby('season')['wins'].idxmax()]
champions['era'] = champions['season'].apply(get_era)

# Count championships per driver
championship_counts = champions['driver_name'].value_counts().reset_index()
championship_counts.columns = ['driver_name', 'championships']

# Get top champions
top_champions = championship_counts.head(15)

fig = px.bar(
    top_champions,
    x='championships',
    y='driver_name',
    orientation='h',
    title='🏆 Most Season Championships (by Race Wins)',
    color='championships',
    color_continuous_scale='Plasma',
    text='championships'
)

fig.update_layout(
    yaxis=dict(autorange="reversed"),
    xaxis_title="Championships Won",
    yaxis_title="Driver",
    height=600,
    template='plotly_dark'
)
fig.show()

In [14]:
# Constructor Championships Over Time
constructor_champs = champions['constructor'].value_counts().reset_index()
constructor_champs.columns = ['constructor', 'championships']

fig = px.pie(
    constructor_champs.head(10),
    values='championships',
    names='constructor',
    title='🏭 Constructor Championship Dominance',
    hole=0.4,
    color_discrete_sequence=px.colors.sequential.RdBu
)

fig.update_layout(
    height=500,
    template='plotly_dark'
)
fig.show()

KeyError: 'constructor'

## 5. 🎯 Fan Recommendations: Must-Watch Historical Races

In [ ]:
# Identify iconic races based on various criteria

# 1. Races with most finishers (close competition)
race_finishes = df[df['position_num'].notna()].groupby(['season', 'race_name']).agg({
    'driver_name': 'count',
    'position_num': 'min'
}).reset_index()
race_finishes.columns = ['season', 'race_name', 'finishers', 'winner_position']

# 2. Races with closest finishes (position 1 and 2 time difference)
# Get winner and runner-up times
winners = df[df['position_num'] == 1][['season', 'race_name', 'driver_name', 'time']].copy()
winners.columns = ['season', 'race_name', 'winner', 'winner_time']

# 3. Most competitive races (many different winners in a season)
season_competition = df[df['is_winner']].groupby('season')['driver_name'].nunique().reset_index()
season_competition.columns = ['season', 'different_winners']
most_competitive_seasons = season_competition.nlargest(10, 'different_winners')

print("🏁 TOP 10 MOST COMPETITIVE SEASONS (Most Different Winners):")
print("="*80)
most_competitive_seasons

In [ ]:
# Create Fan Recommendations
recommendations = {
    "top_5_must_watch_races": [
        {
            "race": "1971 Italian Grand Prix",
            "season": 1971,
            "why": "Closest finish in F1 history - Peter Gethin won by 0.01s",
            "significance": "Monza slipstreaming battle with 5 cars crossing within 0.6s"
        },
        {
            "race": "1984 Monaco Grand Prix",
            "season": 1984,
            "why": "Senna's first win - dominated in torrential rain",
            "significance": "Announced the arrival of a future legend"
        },
        {
            "race": "2008 Brazilian Grand Prix",
            "season": 2008,
            "why": "Hamilton's first championship on final corner of final lap",
            "significance": "Most dramatic championship conclusion ever"
        },
        {
            "race": "2012 Brazilian Grand Prix",
            "season": 2012,
            "why": "Vettel's comeback from last to 6th to win championship",
            "significance": "Overcame first-lap collision in wet conditions"
        },
        {
            "race": "2021 Abu Dhabi Grand Prix",
            "season": 2021,
            "why": "Controversial finish decided championship in final lap",
            "significance": "Verstappen vs Hamilton - the ultimate showdown"
        }
    ],
    "greatest_drivers_by_era": {
        "Pioneer Era (1950-1959)": "Juan Manuel Fangio - 5 Championships in dominant Alfa Romeo and Maserati",
        "Golden Era (1960-1969)": "Jim Clark - Revolutionary driving style, 2 Championships with Lotus",
        "Turbo Era (1970-1979)": "Niki Lauda - Incredible comeback from near-fatal crash",
        "Turbo Era (1980-1989)": "Ayrton Senna - 3 Championships, master of wet conditions",
        "V10 Era (1990-1999)": "Michael Schumacher - First Ferrari champion in 21 years",
        "V10/V8 Era (2000-2009)": "Michael Schumacher - 5 consecutive championships with Ferrari",
        "V8 Era (2010-2013)": "Sebastian Vettel - Youngest 4-time champion in history",
        "Hybrid Era (2014-2021)": "Lewis Hamilton - 7 Championships, most wins in F1 history",
        "Ground Effect Era (2022+)": "Max Verstappen - Dominant back-to-back championships"
    },
    "best_seasons_to_watch": [
        {"season": 1976, "reason": "Hunt vs Lauda - The Rush season"},
        {"season": 1988, "reason": "Senna vs Prost at McLaren - 16 wins from 16 races"},
        {"season": 2007, "reason": "Rookie Hamilton vs Alonso vs Räikkönen"},
        {"season": 2010, "reason": "4-way championship battle to final race"},
        {"season": 2021, "reason": "Verstappen vs Hamilton - Greatest rivalry"}
    ],
    "rising_stars_2024": [
        "Lando Norris - McLaren's championship contender",
        "Oscar Piastri - Rookie sensation matching teammate pace",
        "George Russell - Mercedes future leader",
        "Charles Leclerc - Ferrari's hope for glory",
        "Liam Lawson - Impressive stand-in performances"
    ]
}

# Display recommendations
import json
print("🎯 FAN RECOMMENDATIONS")
print("="*80)
print(json.dumps(recommendations, indent=2))

In [ ]:
# Save recommendations to JSON for website
import json

# Also create key statistics for the website
key_stats = {
    "total_seasons": int(df['season'].nunique()),
    "total_races": int(df[['season', 'race_name']].drop_duplicates().shape[0]),
    "total_drivers": int(df['driver_name'].nunique()),
    "total_constructors": int(df['constructor'].nunique()),
    "most_wins_driver": {
        "name": str(qualified_drivers.index[0]),
        "wins": int(qualified_drivers.iloc[0]['wins'])
    },
    "top_5_drivers": [
        {"name": name, "wins": int(row['wins']), "win_rate": float(row['win_rate'])}
        for name, row in qualified_drivers.head(5).iterrows()
    ],
    "most_successful_constructor": {
        "name": str(df[df['is_winner']]['constructor'].value_counts().index[0]),
        "wins": int(df[df['is_winner']]['constructor'].value_counts().iloc[0])
    },
    "era_breakdown": {
        era: {
            "races": int(stats['total_races']),
            "drivers": int(stats['unique_drivers']),
            "constructors": int(stats['unique_constructors'])
        }
        for era, stats in era_stats.iterrows()
    },
    "goat_top_10": [
        {"name": name, "score": float(row['goat_score']), "wins": int(row['wins'])}
        for name, row in goat_candidates.head(10).iterrows()
    ]
}

# Combine into website data
website_data = {
    "key_stats": key_stats,
    "recommendations": recommendations
}

# Save to JSON
with open('f1_website_data.json', 'w') as f:
    json.dump(website_data, f, indent=2)

print("✅ Website data saved to 'f1_website_data.json'")
print("\n📊 Key Statistics Summary:")
print(f"- Total Seasons: {key_stats['total_seasons']}")
print(f"- Total Races: {key_stats['total_races']}")
print(f"- Total Drivers: {key_stats['total_drivers']}")
print(f"- Most Wins: {key_stats['most_wins_driver']['name']} ({key_stats['most_wins_driver']['wins']} wins)")

## 6. 📊 Additional Visualizations

In [ ]:
# Driver Nationality Distribution
# Extract nationality from driver names (simplified approach)
def guess_nationality(name):
    """Simple nationality guesser based on common patterns"""
    name_lower = name.lower()
    
    # Common patterns (simplified)
    italian_names = ['ascari', 'farina', 'fagioli', 'villoresi', 'trintignant', 'musso', 'collins']
    british_names = ['hamilton', 'moss', 'stewart', 'clark', 'surtees', 'hunt', 'mansell', 'button']
    german_names = ['schumacher', 'vettel', 'rosberg', 'hulkenberg', 'heidfeld']
    brazilian_names = ['senna', 'fittipaldi', 'piquet', 'barrichello', 'massa']
    french_names = ['prost', 'alesi', 'panis', 'trulli', 'grosjean']
    finnish_names = ['hakkinen', 'raikkonen', 'bottas', 'keke rosberg']
    australian_names = ['brabham', 'jones', 'webber', 'ricciardo']
    argentine_names = ['fangio', 'reutemann']
    austrian_names = ['lauda', 'berger', 'wurz']
    dutch_names = ['verstappen']
    spanish_names = ['alonso', 'sainz', 'de la rosa']
    
    if any(n in name_lower for n in italian_names):
        return "Italy"
    elif any(n in name_lower for n in british_names):
        return "United Kingdom"
    elif any(n in name_lower for n in german_names):
        return "Germany"
    elif any(n in name_lower for n in brazilian_names):
        return "Brazil"
    elif any(n in name_lower for n in french_names):
        return "France"
    elif any(n in name_lower for n in finnish_names):
        return "Finland"
    elif any(n in name_lower for n in australian_names):
        return "Australia"
    elif any(n in name_lower for n in argentine_names):
        return "Argentina"
    elif any(n in name_lower for n in austrian_names):
        return "Austria"
    elif any(n in name_lower for n in dutch_names):
        return "Netherlands"
    elif any(n in name_lower for n in spanish_names):
        return "Spain"
    else:
        return "Other"

# Apply to top drivers
top_drivers_nationality = qualified_drivers.head(30).copy()
top_drivers_nationality['nationality'] = top_drivers_nationality.index.map(guess_nationality)

nationality_counts = top_drivers_nationality['nationality'].value_counts().reset_index()
nationality_counts.columns = ['nationality', 'count']

fig = px.pie(
    nationality_counts,
    values='count',
    names='nationality',
    title='🌍 Nationality Distribution (Top 30 Drivers by Wins)',
    color_discrete_sequence=px.colors.qualitative.Set3
)

fig.update_layout(
    height=500,
    template='plotly_dark'
)
fig.show()

In [ ]:
# Career Longevity vs Success
fig = px.scatter(
    qualified_drivers.reset_index(),
    x='career_span',
    y='wins',
    size='races_entered',
    color='win_rate',
    hover_name='driver_name',
    title='📅 Career Longevity vs Success',
    color_continuous_scale='Viridis',
    hover_data=['podiums', 'total_points']
)

# Add trend line
fig.add_trace(go.Scatter(
    x=qualified_drivers['career_span'],
    y=np.polyval(np.polyfit(qualified_drivers['career_span'], qualified_drivers['wins'], 1), 
                qualified_drivers['career_span']),
    mode='lines',
    name='Trend',
    line=dict(color='red', width=2, dash='dash')
))

fig.update_layout(
    xaxis_title="Career Span (Years)",
    yaxis_title="Total Wins",
    height=600,
    template='plotly_dark'
)
fig.show()

## 7. 🏁 Summary & Key Insights

### GOAT Analysis Results:
- **Top GOAT Candidates**: Based on our composite scoring system
- **Key Metrics**: Wins, Win Rate, Podiums, Podium Rate, Total Points, Career Longevity

### Era Evolution:
- **Pioneer Era**: Dominated by Fangio and early constructors
- **Golden Era**: British drivers and Lotus innovation
- **Turbo Era**: Senna vs Prost rivalry
- **Schumacher Era**: Ferrari dominance
- **Modern Era**: Hamilton and Verstappen battles

### Fan Takeaways:
- Most competitive seasons to watch
- Must-see historical races
- Rising stars to follow

---

**Data Source**: F1 Historical Results Dataset  
**Analysis Date**: 2025  
**Tools**: Python, Pandas, Plotly